# 04 - Response Time Analysis

Merge time distributions, time-to-first-review, and trends over time
for PR responsiveness across repos.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oss_pulse.analyze.response_time import (
    compute_first_review_time,
    compute_merge_time_stats,
    merge_time_by_segment,
)
from oss_pulse.visualize.comparison import plot_boxplot_comparison, plot_violin_comparison
from oss_pulse.visualize.style import PALETTE, setup_style

setup_style()

In [ ]:
# Load featured PR data and reviews
DATA_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

pr_df = pd.read_parquet(DATA_DIR / "pr_events_featured.parquet")
reviews_df = pd.read_parquet(RAW_DIR / "pr_reviews.parquet")

print(f"PR events: {len(pr_df)} rows")
print(f"Reviews:   {len(reviews_df)} rows")

In [ ]:
# Per-repo merge time statistics with trend
# TODO: run with real data
merge_stats = compute_merge_time_stats(pr_df)
print("Merge time stats (top 10 fastest):")
merge_stats.sort_values("median_hours").head(10)

In [ ]:
# Merge time distribution histogram
# TODO: run with real data
merged_df = pr_df.dropna(subset=["time_to_merge_hours"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw distribution
axes[0].hist(
    merged_df["time_to_merge_hours"].clip(upper=500),
    bins=50, color=PALETTE["primary"], edgecolor=PALETTE["bg"], alpha=0.85,
)
axes[0].axvline(
    merged_df["time_to_merge_hours"].median(),
    color=PALETTE["accent"], linestyle="--", linewidth=1.2,
    label=f"Median: {merged_df['time_to_merge_hours'].median():.1f}h",
)
axes[0].set_title("Merge Time Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Hours to Merge")
axes[0].set_ylabel("Frequency")
axes[0].legend(loc="upper right")

# Log-scale distribution
axes[1].hist(
    merged_df["time_to_merge_hours"],
    bins=50, color=PALETTE["success"], edgecolor=PALETTE["bg"], alpha=0.85, log=True,
)
axes[1].set_title("Merge Time (Log Scale)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Hours to Merge")
axes[1].set_ylabel("Frequency (log)")

plt.tight_layout()
plt.show()

In [ ]:
# Merge time by PR size bucket
# TODO: run with real data
size_merge = merge_time_by_segment(pr_df, "pr_size_bucket")
print("Merge time by PR size:")
print(size_merge)

fig = plot_boxplot_comparison(
    pr_df.dropna(subset=["time_to_merge_hours"]),
    metric="time_to_merge_hours",
    group="pr_size_bucket",
    title="Merge Time by PR Size",
)
fig.show()

In [ ]:
# Time-to-first-review analysis
# TODO: run with real data
first_review = compute_first_review_time(pr_df, reviews_df)
print("First review time stats:")
first_review.sort_values("median_hours")

In [ ]:
# Trend in merge times over months
# TODO: run with real data
monthly_df = pd.read_parquet(DATA_DIR / "repo_monthly.parquet")

overall_monthly = (
    monthly_df.groupby(["year", "month"])["median_merge_time_hours"]
    .median()
    .reset_index()
)
overall_monthly["date"] = pd.to_datetime(
    overall_monthly["year"].astype(str) + "-" + overall_monthly["month"].astype(str).str.zfill(2) + "-01"
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    overall_monthly["date"], overall_monthly["median_merge_time_hours"],
    color=PALETTE["primary"], linewidth=1.5, marker="o", markersize=3,
)
ax.set_title("Median Merge Time Trend (All Repos)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Median Merge Time (hours)")
plt.tight_layout()
plt.show()